In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# pytorch dataset
import torch # 파이토치라는 딥러닝 프레임워크 라이브러리입니다. 딥러닝 모델을 만들고 학습시키는 데 사용
from torch.utils.data import Dataset, DataLoader # 데이터를 모델에 넣기 좋게 관리하고, 미니배치(학습할 때 쓰는 데이터 묶음) 단위로 나눠서 불러오는 데 사용

# dataset split
from sklearn.model_selection import train_test_split

# model evaluate
from sklearn.metrics import f1_score
# f1_score: 모델이 얼마나 잘 맞추는지 평가하는 데 사용

# model
import torch.nn as nn # 딥러닝 모델을 쉽게 만들 수 있도록 도와주는 모듈 (CNN 모델 만들 때 사용)
import torch.nn.functional as F # 신경망(NN)에서 자주 쓰는 함수(활성화 함수(activation func), 손실 함수(loss func 등)를 제공

# progress bar
from tqdm import tqdm # 모델 학습 진행률 상황 시각화

# counter
from collections import Counter

import torch.optim as optim

# 혼동행렬
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## 1. load dataset

In [ ]:
# 1. 데이터 불러오기
df = pd.read_csv('/content/drive/MyDrive/네바다 주립 대학/프로젝트/uniprotkb_AND_reviewed_true_2025_06_25.tsv', sep='\t')

## 2. EC number 및 시퀀스 기본 정제

In [ ]:
# EC 번호 없는 결측치 제거
df = df[df['EC number'].notna()]

# EC number 중 '-'와 'n'이 포함된 행 제거
df = df[~df['EC number'].str.contains('-', regex=False)]
df = df[~df['EC number'].str.contains('n', regex=False)]

In [ ]:
def extract_2nd_level(ec_string):
    try:
        ec_list = ec_string.split(';')
        ec_trimmed = []
        for ec in ec_list:
            parts = ec.strip().split('.')
            if len(parts) >= 2 and parts[0] == '1':  # 첫 번째 클래스가 1인 경우만 예시
                trimmed = '.'.join(parts[:2])
                ec_trimmed.append(trimmed)
        # 중복 제거
        return sorted(set(ec_trimmed))
    except:
        return []

df['ec_2nd_level'] = df['EC number'].apply(extract_2nd_level)

# 2. ec_2nd_level이 빈 리스트인 행(즉, 1로 시작하는 EC 넘버가 하나도 없는 행) 삭제
df = df[df['ec_2nd_level'].apply(lambda x: len(x) > 0)].reset_index(drop=True)

In [ ]:
# ✅ 사용할 아미노산 문자 집합 정의
# 표준 20종 + 모호하지만 자주 등장하는 'X' 포함
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWYX'
# 제거하거나 대체할 문자는 별도 정의
INVALID_AMINO_ACIDS = 'BZOUJ'

def is_valid_sequence(seq):
    """
    서열이 허용된 아미노산 문자, 또는 INVALID_AMINO_ACIDS로만 구성됐는지 검사.
    그 외 문자가 있으면 False 반환.
    """
    seq = seq.upper().strip()
    for aa in seq:
        if aa not in AMINO_ACIDS and aa not in INVALID_AMINO_ACIDS:
            return False
    return True

def clean_sequence(seq):
    """
    B, Z, O, U, J → X로 치환, 나머지는 그대로.
    """
    seq = seq.upper().strip()
    cleaned = ''
    for aa in seq:
        if aa in AMINO_ACIDS:
            cleaned += aa
        elif aa in INVALID_AMINO_ACIDS:
            cleaned += 'X'
    return cleaned

# ✅ 시퀀스 정제 적용
df = df[df['Sequence'].apply(is_valid_sequence)].copy()
df['cleaned_seq'] = df['Sequence'].apply(clean_sequence)

In [ ]:
# 중복되는 시퀀스 중 ec 넘버가 다른 경우 처리
# cleaned_seq 기준으로 ec_2nd_level을 합친 딕셔너리 생성
seq_to_ec = (
    df.groupby('cleaned_seq')['ec_2nd_level']
    .apply(lambda x: sorted(set(sum(x, []))))
    .to_dict()
)

# ec_2nd_level 컬럼을 합쳐진 값으로 덮어쓰기
df['ec_2nd_level'] = df['cleaned_seq'].map(seq_to_ec)

In [ ]:
# ✅ 중복 서열 제거 (동일한 시퀀스만 유지)
df = df.drop_duplicates(subset='cleaned_seq').reset_index(drop=True)

In [ ]:
all_ecs_flat = [ec for ecs in df['ec_2nd_level'] for ec in ecs]
ec_counts = Counter(all_ecs_flat)

rare_ecs = set([ec for ec, cnt in ec_counts.items() if cnt <= 10])

def has_rare_ec(ec_list):
    return any(ec in rare_ecs for ec in ec_list)

df = df[~df['ec_2nd_level'].apply(has_rare_ec)].reset_index(drop=True)

In [ ]:
# 고유 클래스 추출
# 클래스를 중복되는 번호 없이 나열하여 2단계 클래스만은 담은 집합 만들기
all_ecs_2nd = set()
df['ec_2nd_level'].apply(lambda x: [all_ecs_2nd.add(ec) for ec in x])

# 2. 숫자 기반 정렬 함수 정의
def ec_sort_key(ec):
    return tuple(int(part) for part in ec.split('.'))

# 3. 숫자 정렬 적용
all_ecs_2nd = sorted(list(all_ecs_2nd), key=ec_sort_key)

# 4. 인덱스 매핑
ec2_to_idx = {ec: i for i, ec in enumerate(all_ecs_2nd)}
idx_to_ec2 = {i: ec for ec, i in ec2_to_idx.items()}


In [ ]:
# 각 고유 클래스(여기선 EC 2단계 번호)에 숫자 인덱스를 붙여주는 작업
# 클래스 multilabel vactor 만드는 것

def encode_multilabel_2nd_level(ec_list):
    vec = np.zeros(len(ec2_to_idx), dtype=int)
    for ec in ec_list:
        if ec in ec2_to_idx:
            vec[ec2_to_idx[ec]] = 1
    return vec

df['label_vector'] = df['ec_2nd_level'].apply(encode_multilabel_2nd_level)

In [ ]:
print(df.head(5))
print("남은 EC 2단계 클래스 수:", len(ec2_to_idx))
print("남은 데이터 샘플 수:", len(df))

        Entry   Entry Name  EC number  \
0  A0A059TC02   CCR1_PETHY   1.2.1.44   
1  A0A075BSX9   HLNO_SHIS7    1.5.3.5   
2  A0A080WMA9   DHAS_TRIRC   1.2.1.11   
3  A0A087X1C5  CP2D7_HUMAN  1.14.14.1   
4  A0A095C6S0  OXDA2_CRYD2    1.4.3.3   

                                            Sequence ec_2nd_level  \
0  MRSVSGQVVCVTGAGGFIASWLVKILLEKGYTVRGTVRNPDDPKNG...        [1.2]   
1  MTEKIYDAIVVGAGFSGLVAARELSAQGRSVLIIEARHRLGGRTHV...        [1.5]   
2  MATPSKKRCGVLGATGAVGTRFILLLSQHPLLELVAVGASDRSSGK...        [1.2]   
3  MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...       [1.14]   
4  MSFDAVVIGSGVIGLSIARELDNRGLKVAMVARDLAEDSLSVGFAS...        [1.4]   

                                         cleaned_seq  \
0  MRSVSGQVVCVTGAGGFIASWLVKILLEKGYTVRGTVRNPDDPKNG...   
1  MTEKIYDAIVVGAGFSGLVAARELSAQGRSVLIIEARHRLGGRTHV...   
2  MATPSKKRCGVLGATGAVGTRFILLLSQHPLLELVAVGASDRSSGK...   
3  MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...   
4  MSFDAVVIGSGVIGLSIARELDNRGLKVAMVARDLAEDSLSVGFAS...   

 

In [ ]:
print(ec2_to_idx)
print(len(ec2_to_idx))

{'1.1': 0, '1.2': 1, '1.3': 2, '1.4': 3, '1.5': 4, '1.6': 5, '1.7': 6, '1.8': 7, '1.9': 8, '1.10': 9, '1.11': 10, '1.12': 11, '1.13': 12, '1.14': 13, '1.15': 14, '1.16': 15, '1.17': 16, '1.18': 17, '1.20': 18, '1.21': 19, '1.23': 20, '1.97': 21}
22


In [ ]:
# ec_2nd_level 컬럼을 explode하여 분리
exploded = df.explode('ec_2nd_level')

# 각 EC 번호별 개수 계산 및 정렬
ec_counts = exploded['ec_2nd_level'].value_counts().sort_index()

print(ec_counts)

ec_2nd_level
1.1     6623
1.10     439
1.11    1107
1.12     105
1.13     689
1.14    2580
1.15     410
1.16     186
1.17    1638
1.18     462
1.2     3014
1.20      66
1.21     149
1.23      17
1.3     2462
1.4     1256
1.5     1221
1.6      503
1.7      814
1.8     1186
1.9      103
1.97     372
Name: count, dtype: int64


In [ ]:
multi_label_count = df[df['ec_2nd_level'].apply(lambda x: len(x) > 1)].shape[0]

print(f"멀티라벨 데이터 수: {multi_label_count}")

멀티라벨 데이터 수: 192


## 3. 시퀀스 3가지 방식(one-hot, k-mer, token) 프로세싱

### 3-1. 원핫 인코딩

In [ ]:
class SeqEncoder:
    def __init__(self):
        self.categories = np.array(['PAD'] + list('ACDEFGHIKLMNPQRSTVWYX'))
        self.char_to_idx = {char: idx for idx, char in enumerate(self.categories)}

    def char_to_one_hot_encoding(self, c):
        X_int = np.zeros(len(self.categories), dtype=np.int8)
        if c in self.char_to_idx:
            X_int[self.char_to_idx[c]] = 1
        return X_int


    def seq_to_one_hot_encoding(self, seq):
        return np.array([self.char_to_one_hot_encoding(c) for c in seq])

    def one_hot_encoding_to_seq(self, one_hot_encoding):
        X = np.array(one_hot_encoding).reshape(-1, 22)
        last_index = np.where(np.sum(X, axis=1) == 0)[0]
        last_index = 1000 if len(last_index) == 0 else last_index[0]
        return ''.join(self.categories[X.argmax(axis=1)][:last_index])

# ⚙️ 전처리 함수
def add_one_hot_column(df, max_len=512):
    encoder = SeqEncoder()

    def process(seq):
        one_hot = encoder.seq_to_one_hot_encoding(seq)
        if len(one_hot) >= max_len:
            return one_hot[:max_len]
        else:
            pad_len = max_len - len(one_hot)
            return np.vstack([one_hot, np.zeros((pad_len, 22), dtype=np.int8)])

    df['encoded_onehot'] = df['cleaned_seq'].apply(process)
    return df

In [ ]:
# 원핫 인코딩
df = add_one_hot_column(df)

### 3-2. k-mer 인코딩

In [ ]:
# K-mer 방식

K = 3  # 사용할 K-mer 길이

# 1. K-mer 추출 함수
def generate_kmers(seq, k=K):
    seq = seq.upper().strip()
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]

# 2. 전체 데이터에서 K-mer 모으기
all_kmers = Counter()
for seq in df['cleaned_seq']:
  all_kmers.update(generate_kmers(seq))

In [ ]:
# all_kmers: Counter 객체라고 가정
all_unique_kmers = list(all_kmers.keys())  # 전체 K-mer 리스트 추출

# 인덱스 부여 (PAD=0, UNK=1 이후 2부터 시작)
kmer_to_idx = {kmer: idx + 2 for idx, kmer in enumerate(all_unique_kmers)}
kmer_to_idx['PAD'] = 0
kmer_to_idx['UNK'] = 1

In [ ]:
# kmer 시퀀스 인코딩
def encode_kmer_sequence(seq, kmer_to_idx, max_len=512, k=3):
    kmers = generate_kmers(seq, k)
    encoded = []

    for kmer in kmers:
        if kmer in kmer_to_idx:
            encoded.append(kmer_to_idx[kmer])
        else:
            encoded.append(kmer_to_idx['UNK'])

    # 자르거나 PAD로 채우기
    if len(encoded) > max_len:
        encoded = encoded[:max_len]
    else:
        encoded += [kmer_to_idx['PAD']] * (max_len - len(encoded))

    return encoded

# 시퀀스 인코딩 적용
df['encoded_kmers'] = df['cleaned_seq'].apply(lambda x: encode_kmer_sequence(x, kmer_to_idx, max_len=512))

### 3-3. 정수형 token 인코딩

In [ ]:
# ✅ 아미노산 문자 → 정수 인덱스 매핑 (PAD=0)
aa_to_idx = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}  # 1부터 시작
aa_to_idx['PAD'] = 0  # 패딩은 0으로 고정

# ✅ 정수 인코딩 함수
def encode_sequence(seq, max_len=512, aa_to_idx=None):
    """
    정제된 시퀀스를 정수 인덱스 리스트로 변환하고 길이를 맞춰줍니다.
    - 길이 초과: 잘라냄
    - 길이 부족: PAD(0)로 채움
    """
    encoded = [aa_to_idx.get(aa, aa_to_idx['X']) for aa in seq]

    # 시퀀스 길이 맞추기
    if len(encoded) > max_len:
        encoded = encoded[:max_len]
    else:
        encoded += [aa_to_idx['PAD']] * (max_len - len(encoded))

    return encoded

In [ ]:
# 기존 시퀀스 데이터(문자형)를 변환(정수형)하여 데이터 셋에 추가
df['encoded_token'] = df['Sequence'].apply(lambda s: encode_sequence(s, max_len=512, aa_to_idx=aa_to_idx))

In [ ]:
# 변환한 시퀀스 데이터 확인
df.head(5)

,Entry,Entry Name,EC number,Sequence,ec_2nd_level,cleaned_seq,label_vector,encoded_onehot,encoded_kmers,encoded_token
0,A0A059TC02,CCR1_PETHY,1.2.1.44,MRSVSGQVVCVTGAGGFIASWLVKILLEKGYTVRGTVRNPDDPKNG...,[1.2],MRSVSGQVVCVTGAGGFIASWLVKILLEKGYTVRGTVRNPDDPKNG...,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[11, 15, 16, 18, 16, 6, 14, 18, 18, 2, 18, 17,..."
1,A0A075BSX9,HLNO_SHIS7,1.5.3.5,MTEKIYDAIVVGAGFSGLVAARELSAQGRSVLIIEARHRLGGRTHV...,[1.5],MTEKIYDAIVVGAGFSGLVAARELSAQGRSVLIIEARHRLGGRTHV...,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,...","[314, 315, 316, 317, 318, 319, 320, 321, 322, ...","[11, 17, 4, 9, 8, 20, 3, 1, 8, 18, 18, 6, 1, 6..."
2,A0A080WMA9,DHAS_TRIRC,1.2.1.11,MATPSKKRCGVLGATGAVGTRFILLLSQHPLLELVAVGASDRSSGK...,[1.2],MATPSKKRCGVLGATGAVGTRFILLLSQHPLLELVAVGASDRSSGK...,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,...","[707, 708, 709, 710, 711, 712, 713, 714, 715, ...","[11, 1, 17, 13, 16, 9, 9, 15, 2, 6, 18, 10, 6,..."
3,A0A087X1C5,CP2D7_HUMAN,1.14.14.1,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,[1.14],MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,...","[1002, 284, 639, 640, 1003, 1004, 809, 390, 10...","[11, 6, 10, 4, 1, 10, 18, 13, 10, 1, 11, 8, 18..."
4,A0A095C6S0,OXDA2_CRYD2,1.4.3.3,MSFDAVVIGSGVIGLSIARELDNRGLKVAMVARDLAEDSLSVGFAS...,[1.4],MSFDAVVIGSGVIGLSIARELDNRGLKVAMVARDLAEDSLSVGFAS...,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,...","[1367, 1368, 625, 748, 838, 174, 98, 1369, 137...","[11, 16, 5, 3, 1, 18, 18, 8, 6, 16, 6, 18, 8, ..."


## 4. 모델 생성

### 4-1. PyTorch Dataset 생성

In [ ]:
class ProteinDataset(Dataset):
    # 데이터를 메모리에 올려두고, 인덱스로 쉽게 접근할 수 있게 준비
    def __init__(self, dataframe, mode='token'):
        """
        mode: 'token', 'kmer', 'onehot'
        """
        # 전처리 방식에 따라 시퀀스 열 선택
        if mode == 'token':
            self.sequences = dataframe['encoded_token'].tolist() # 숫자로 변환된 단백질 서열
        elif mode == 'kmer':
            self.sequences = dataframe['encoded_kmers'].tolist() # kmer로 변환된 단백질 서열
        elif mode == 'onehot':
            self.sequences = dataframe['encoded_onehot'].tolist() # 원핫으로 변환된 단백질 서열
        else:
            raise ValueError(f"Unknown mode: {mode}")

        self.labels = dataframe['label_vector'].tolist()
        self.mode = mode  # token / kmer / onehot

    def __len__(self):
        return len(self.sequences)  # 전체 데이터(서열)의 개수를 반환

    def __getitem__(self, idx):
        # onehot은 FloatTensor, 나머지는 LongTensor
        if self.mode == 'onehot':
            x = torch.FloatTensor(self.sequences[idx]) # 서열(원핫 인코딩)을 FloatTensor(실수형 텐서)로 변환
        else:
            x = torch.LongTensor(self.sequences[idx]) # 서열(정수 시퀀스)을 LongTensor(정수형 텐서)로 변환

        y = torch.FloatTensor(self.labels[idx]) # 다중(multi-hot) 라벨을 FloatTensor(실수형 텐서, 멀티라벨용)로 변환
        return x, y


# 파이토치로 데이터셋을 정리하는 이유
# - PyTorch의 DataLoader를 사용하면 모델 학습 시 자동으로 데이터를 배치 단위로 불러옴
# - 라벨을 텐서(Tensor)로 변환하면 GPU에서도 빠르게 계산한 가능

### 4-2. 모델 구조 생성

In [ ]:
class ProteinCNN(nn.Module): # 모델 클래스 생성
  def __init__(self, mode, vocab_size, embed_dim, num_classes, seq_len=512, kernel_size=5):
    # vocab_size: 아미노산 알파벳(문자) 개수 + 특수토큰 개수
    # embed_dim: 임베딩(embedding) 차원(아미노산을 어떤 크기의 벡터로 바꿀지)
    # num_classes: 예측할 클래스(라벨) 개수
    # kernel_size: 합성곱 필터 크기(기본값 5)

    super(ProteinCNN, self).__init__()

    self.mode = mode  # 'token' | 'onehot' | 'kmer'
    self.seq_len = seq_len

    if self.mode in ['token', 'kmer']:
      # 각 아미노산 문자를 고정된 크기의 벡터(숫자 배열)로 변환
      self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)
      in_channels = embed_dim
    elif self.mode == 'onehot':
      # 원핫 인코딩: embedding 제거
      self.embedding = None
      in_channels = vocab_size
    else:
      raise ValueError("mode must be 'token', 'kmer', or 'onehot'")

    # CNN 레이어 구성

    self.conv1 = nn.Conv1d(in_channels=in_channels, out_channels=128, kernel_size=kernel_size, padding=2) # 단백질 시퀀스에서 연속된 패턴 탐색
    self.pool1 = nn.MaxPool1d(kernel_size=2) # 시퀸스 주요 데이터 추출

    # hidden layer 2
    self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=kernel_size, padding=2)
    self.pool2 = nn.MaxPool1d(kernel_size=2)

    self.dropout = nn.Dropout(0.5) # 히든 레이어에 적용하는 정규화

    # output layer
    # CNN에서 추출된 정보를 최종적으로 클래스(라벨)별 점수로 변환
    self.fc = nn.Linear(256 * (512 // 4), num_classes)  # 512 길이 → 2번 pool → 512/4(128개)
    # 512 → MaxPool(2) → 256 → MaxPool(2) → 128

  # 실제 데이터가 모델을 통과하는 과정
  def forward(self, x):
    # x: [batch, seq_len] or [batch, seq_len, vocab_size]
    if self.mode in ['token', 'kmer']:
      x = self.embedding(x)             # [B, L, D]
      x = x.permute(0, 2, 1)            # [B, D, L]
    elif self.mode == 'onehot':
      x = x.permute(0, 2, 1)            # [B, vocab_size, L]

    x = F.relu(self.conv1(x))   # → [batch, 128, seq_len]
    x = self.pool1(x)           # → [batch, 128, seq_len/2]

    x = F.relu(self.conv2(x))   # → [batch, 256, seq_len/2]
    x = self.pool2(x)           # → [batch, 256, seq_len/4]

    x = x.view(x.size(0), -1)   # flatten을 사용해 여러 차원을 하나로 펴줌
    x = self.dropout(x)
    return self.fc(x)           # → [batch, num_classes]

### 4-3. 학습 평가 생성

In [ ]:
def evaluate(model, val_loader, criterion, device, threshold=0.2):
    model.eval()    # 모델을 평가(evaluation) 모드 전환
    total_loss = 0  # 전체 손실값
    all_preds = []  # 전체 예측값
    all_labels = [] # 전체 실제 정답값

    with torch.no_grad(): # 메모리와 속도를 아끼기 위해 기울기 계산 off
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device) # 데이터를 GPU로 가동

            outputs = model(x_batch) # 모델에 입력해서 예측값(outputs, 점수)을 획득

            # 손실값 계산 및 누적
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()

            # 예측값 (sigmoid → binary prediction)
            probs = torch.sigmoid(outputs)                  # 점수를 확률(0~1)로 변환 (sigmoid, activation func)
            preds = (probs > threshold).int().cpu().numpy() # threshold(0.2 값으로 설정) 넘으면 1, 아니면 0
            labels = y_batch.cpu().numpy()                  # 실제 정답값도 numpy로 변환

            all_preds.append(preds)   # 예측 결과 저장
            all_labels.append(labels) # 정답 결과 저장

    # 리스트 → 전체 배열로 결합
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    avg_loss = total_loss / len(val_loader)
    print(f"🔵 Validation Loss: {avg_loss:.4f}\n")

### 4-4. 학습 루프 생성

In [ ]:
# 모델을 학습(training)하고, 매 에폭(epoch)마다 검증(validation) 데이터로 성능을 평가하는 전체 과정을 자동으로 실행

def train_model(model, train_loader, val_loader, optimizer, criterion, device, num_epochs=10):
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        # 🔄 학습 루프
        # train_loader에서 데이터를 batch_size만큼씩 꺼내서 학습
        for x_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device) # 데이터를 GPU로 가동

            optimizer.zero_grad()     # 이전 기울기(gradient) 초기화
            outputs = model(x_batch)  # 모델에 입력값 넣고 예측값(logits) 얻기
            loss = criterion(outputs, y_batch)    # 예측값과 정답 비교해 손실(loss) 계산
            loss.backward()           # 손실을 기준으로 기울기(gradient) 계산
            optimizer.step()          # 파라미터(가중치) 업데이트

            total_loss += loss.item() # 각 배치별 손실값을 더해서, 전체 평균 손실을 계산

        # 한 에폭이 끝날 때마다 평균 손실을 출력
        avg_loss = total_loss / len(train_loader)
        print(f"🟢 Epoch {epoch+1}")
        print(f"🟠 Train Loss: {avg_loss:.4f}")

        # 매 에폭이 끝날 때마다 검증 데이터로 모델의 성능을 평가
        evaluate(model, val_loader, criterion, device)

In [ ]:
# 각 EC 넘버별 성능 취합
def get_classwise_scores(y_true, y_pred, class_names):
    f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    return pd.DataFrame({
        'Class': class_names,
        'F1-score': f1
    })


# 각 프로세싱 방식별 f1 스코어(macro, micro) 도출
def get_macro_micro_f1(y_true, y_pred):
    macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
    return macro, micro

### 4-5 모델 평가 지표 시각화

In [ ]:
# 예측값/정답을 따로 모아서, 다양한 방식으로 분석하고 가능하도록 하는 함수
def evaluate_model(model, data_loader, device):
    model.eval()    # 모델을 평가(evaluation) 모드 전환
    all_preds = []  # 전체 예측값
    all_labels = [] # 전체 실제 정답값

    with torch.no_grad():
        # 데이터로더에서 데이터를 한 배치씩 꺼내서 GPU/CPU로 옮김
        for batch in data_loader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)

            # 모델 예측 및 이진화
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.2).int()  # 🔸 threshold 조정 가능

            all_labels.append(labels.cpu())
            all_preds.append(preds.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_preds = torch.cat(all_preds).numpy()

    # 전체 정답과 전체 예측값을 반환
    return all_labels, all_preds

## 5. 모델 학습

In [ ]:
# 각 모드별로 결과 저장
macro_micro_table = []
models = {}
f1_tables = []


for mode in ['onehot', 'kmer', 'token']:
    print(f"\n===== [모드: {mode}] =====")

    # Dataset 분할
    df_trainval, df_test = train_test_split(df, test_size=0.2, random_state=42)
    df_train, df_val = train_test_split(df_trainval, test_size=0.2, random_state=42)

    # PyTorch Dataset
    train_dataset = ProteinDataset(df_train, mode=mode)
    val_dataset = ProteinDataset(df_val, mode=mode)
    test_dataset = ProteinDataset(df_test, mode=mode)

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # CNN 모델 초기화
    if mode == 'onehot':
        vocab_size = len(aa_to_idx)  # aa_to_idx: 아미노산+특수토큰 사전
        model = ProteinCNN(mode='onehot', vocab_size=vocab_size, embed_dim=None, num_classes=len(ec2_to_idx))
    elif mode == 'token':
        model = ProteinCNN(mode='token', vocab_size=len(aa_to_idx), embed_dim=32, num_classes=len(ec2_to_idx))
    elif mode == 'kmer':
        vocab_size = max([max(seq) for seq in df['encoded_kmers']]) + 1
        model = ProteinCNN(mode='kmer', vocab_size=vocab_size, embed_dim=32, num_classes=len(ec2_to_idx))

    # 모델 학습
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 모델의 예측값과 정답(라벨) 사이의 차이(오차)를 계산
    criterion = nn.BCEWithLogitsLoss()     # multi-label 손실

    # 모델의 파라미터(가중치)를 업데이트 방식 지정, lr: 학습률(learning rate)은 0.001로 설정
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_model(model, train_loader, val_loader, optimizer, criterion, device, num_epochs=10)
    # 모델 학습
    # - 위에서 준비한 모델, 데이터, 손실 함수, 옵티마이저, 연산 장치 등을 모두 넣고, 10번(epoch) 동안 학습을 진행
    # - 학습이 진행되는 동안, 매 에폭마다 검증 데이터로 성능도 같이 평가

    models[mode] = model

    all_labels, all_preds = evaluate_model(model, test_loader, device)
    class_names = list(ec2_to_idx.keys())

    # ------ 🔔 혼동 행렬 출력부 추가 ------
    # test_loader에 검증/테스트셋만 들어있는지 재확인
    y_true_flat = all_labels.flatten()
    y_pred_flat = all_preds.flatten()

    # 2. 이진 혼동 행렬 계산 (전체 클래스 기준)
    cm = confusion_matrix(y_true_flat, y_pred_flat, labels=[0, 1])
    print("전체 2x2 혼동행렬 (TN, FP / FN, TP):")
    print(cm)

    # 3. 혼동 행렬 시각화
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative (0)", "Positive (1)"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"[{mode}] Overall 2x2 Confusion_matrix")
    plt.show()
    # ---------------------------------

    df_f1 = get_classwise_scores(all_labels, all_preds, class_names)
    df_f1 = df_f1.set_index('Class')
    df_f1.rename(columns={'F1-score': f'F1-{mode}'}, inplace=True)
    f1_tables.append(df_f1)

    macro_f1, micro_f1 = get_macro_micro_f1(all_labels, all_preds)
    macro_micro_table.append({
        "Mode": mode,
        "Macro F1-score": macro_f1,
        "Micro F1-score": micro_f1
    })

## 6. 모델 평가 시각화

In [ ]:
# 각 ec 넘버별 성능
df_f1_all = pd.concat(f1_tables, axis=1)
df_f1_all.reset_index(inplace=True)
print(df_f1_all)

In [ ]:
# df_f1_all: Class, F1-onehot, F1-kmer, F1-token 열이 있는 DataFrame
bar_width = 0.25
indices = np.arange(len(df_f1_all))

plt.figure(figsize=(18, 6))
plt.bar(indices - bar_width, df_f1_all['F1-onehot'], width=bar_width, label='One-hot')
plt.bar(indices, df_f1_all['F1-kmer'], width=bar_width, label='K-mer')
plt.bar(indices + bar_width, df_f1_all['F1-token'], width=bar_width, label='Token')

plt.xlabel('EC Class')
plt.ylabel('F1-score')
plt.title('each EC numbers F1-score (One-hot vs K-mer vs Token)')
plt.xticks(indices, df_f1_all['Class'], rotation=90)
plt.legend()
plt.tight_layout()
plt.grid(axis='y')
plt.show()


In [ ]:
# 평균 F1-score 계산
mean_f1_scores = df_f1_all[['F1-onehot', 'F1-kmer', 'F1-token']].mean()
print(mean_f1_scores)

In [ ]:
# 모든 모드별 F1-score를 하나의 표로 합침
df_macro_micro = pd.DataFrame(macro_micro_table)
print(df_macro_micro)

## 7. 예측

In [ ]:
def predict_ec_number(
    seq,
    mode,
    models,
    ec2_to_idx,
    aa_to_idx=None,
    kmer_to_idx=None,
    max_len=512,
    k=3,
    threshold=0.2,
    device='cpu'
):
    """
    단일 시퀀스와 인코딩 모드(onehot/kmer/token)를 받아 해당 모델로 EC 넘버 예측
    """
    import numpy as np
    import torch

    model = models[mode]
    model.eval()

    # 인코딩
    if mode == 'token':
        encoded = encode_sequence(seq, max_len=max_len, aa_to_idx=aa_to_idx)
        input_tensor = torch.LongTensor(np.array([encoded])).to(device)

    elif mode == 'kmer':
        # 함수 정의에 맞게 인자 개수만 전달
        # 예시: def encode_kmer_sequence(seq, kmer_to_idx, max_len)
        encoded = encode_kmer_sequence(seq, kmer_to_idx, max_len)
        if isinstance(encoded, dict):
            raise ValueError("encode_kmer_sequence()는 리스트 또는 배열을 반환해야 합니다.")
        input_tensor = torch.LongTensor(np.array([encoded])).to(device)

    elif mode == 'onehot':
        encoder = SeqEncoder()
        one_hot = encoder.seq_to_one_hot_encoding(seq)
        if len(one_hot) >= max_len:
            one_hot = one_hot[:max_len]
        else:
            pad_len = max_len - len(one_hot)
            one_hot = np.vstack([one_hot, np.zeros((pad_len, one_hot.shape[1]), dtype=np.int8)])
        input_tensor = torch.FloatTensor(np.expand_dims(one_hot, axis=0)).to(device)

    else:
        raise ValueError("지원하지 않는 모드입니다. 'token', 'kmer', 'onehot' 중 선택하세요.")

    # 예측
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.sigmoid(output).cpu().numpy()[0]
        predicted_classes = [
            ec for ec, idx in ec2_to_idx.items() if probs[idx] > threshold
        ]
    return predicted_classes


In [ ]:
# 예측 대상 시퀀스
test_seq = "MSETDMKLVVVGAAGRMGQTLIRTVHEAAGVRLHAAIERSNSPFIGRDAGELAGLGPIGVPITDKPLEAFVEAEGVLDFTAPAGTVEFAGLAAQARIVHVIGTTGCSADDEAKIRAAARHARVVKSGNMSLGVNLLGVLTETAARALSAKDWDIEILEMHHRHKVDAPSGTALLLGEAAAKGRGIDLADQAVKVRDGHTGPRPQGTIGFATLRGGLVVGEHSVILAGEGELVTLSHSATDRSIFARGAVAAALWGRSQKPGFYTMLDVLGLN"

# 각 모드별 예측
for mode in ['onehot', 'kmer', 'token']:
    predicted_ecs = predict_ec_number(
        seq=test_seq,
        mode=mode,
        models=models,
        ec2_to_idx=ec2_to_idx,
        aa_to_idx=aa_to_idx,
        kmer_to_idx=kmer_to_idx,
        max_len=512,
        k=3,
        threshold=0.2,
        device=device
    )
    print(f"[{mode}] 예측된 EC 번호:", predicted_ecs)
